# Phase 1: Data Exploration & Understanding
## Avoidable Emergency Department Utilization Navigator

This notebook performs exploratory data analysis on the real CMS Medicare Synthetic Enrollment and FFS Claims Datasets (`beneficiary_2022.csv`, `inpatient.csv`, and `outpatient.csv`).

### Data Summary:
- `beneficiary_2022.csv`: 8,673 beneficiaries (Member demographics, enrollment, dual status)
- `inpatient.csv`: 58,068 claims (Inpatient & Emergency encounters, ICD-10 diagnoses, DRG codes, revenue center `0450` ED charges)
- `outpatient.csv`: 575,094 claims (Outpatient, Urgent Care, and ED outpatient encounters, revenue center `0450` ED charges)

**Delimiter Note**: CMS claims files use pipe (`|`) delimiter.

### Step 1: Load Datasets

In [ ]:
import pandas as pd
import numpy as np

# File paths
BENEFICIARY_PATH = '../datasets/beneficiary_2022.csv'
INPATIENT_PATH = '../datasets/inpatient.csv'
OUTPATIENT_PATH = '../datasets/outpatient.csv'

# Load datasets with pipe delimiter
df_bene = pd.read_csv(BENEFICIARY_PATH, sep='|', low_memory=False)
df_inp = pd.read_csv(INPATIENT_PATH, sep='|', low_memory=False)
df_outp = pd.read_csv(OUTPATIENT_PATH, sep='|', low_memory=False)

print(f"Beneficiary Dataset Shape: {df_bene.shape}")
print(f"Inpatient Claims Dataset Shape: {df_inp.shape}")
print(f"Outpatient Claims Dataset Shape: {df_outp.shape}")

### Step 2: Display Dataset Profiles (Shape, Dtypes, Missing Values, Sample Records)

In [ ]:
print("=== Beneficiary Dataset Profile ===")
print(f"Rows: {df_bene.shape[0]}, Columns: {df_bene.shape[1]}")
print("Top 10 Null Counts:")
print(df_bene.isnull().sum().sort_values(ascending=False).head(10))
df_bene[['BENE_ID', 'STATE_CODE', 'COUNTY_CD', 'ZIP_CD', 'BENE_BIRTH_DT', 'SEX_IDENT_CD', 'BENE_RACE_CD', 'AGE_AT_END_REF_YR']].head()

In [ ]:
print("=== Inpatient Claims Dataset Profile ===")
print(f"Rows: {df_inp.shape[0]}, Columns: {df_inp.shape[1]}")
print("Top 10 Null Counts:")
print(df_inp.isnull().sum().sort_values(ascending=False).head(10))
df_inp[['BENE_ID', 'CLM_ID', 'CLM_FROM_DT', 'CLM_THRU_DT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'ADMTG_DGNS_CD', 'PRNCPAL_DGNS_CD', 'REV_CNTR']].head()

In [ ]:
print("=== Outpatient Claims Dataset Profile ===")
print(f"Rows: {df_outp.shape[0]}, Columns: {df_outp.shape[1]}")
print("Top 10 Null Counts:")
print(df_outp.isnull().sum().sort_values(ascending=False).head(10))
df_outp[['BENE_ID', 'CLM_ID', 'CLM_FROM_DT', 'CLM_THRU_DT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'PRNCPAL_DGNS_CD', 'REV_CNTR']].head()

### Step 3: Explanation of Important Columns

| Column Name | CMS Definition / Healthcare Meaning |
| :--- | :--- |
| `BENE_ID` | Beneficiary ID / Unique Member Key |
| `BENE_BIRTH_DT` | Date of Birth (DD-MMM-YYYY) |
| `SEX_IDENT_CD` | Gender Code (1 = Male, 2 = Female) |
| `AGE_AT_END_REF_YR` | Calculated Age at End of Reference Year |
| `CLM_ID` | Unique Claim Header Identifier |
| `CLM_FROM_DT`, `CLM_THRU_DT` | Start and end dates of service encounter |
| `PRVDR_NUM` | CMS Certification Number of Billing Facility/Hospital |
| `AT_PHYSN_NPI` | Attending Physician NPI |
| `CLM_PMT_AMT` | Total Medicare Claim Reimbursement Amount ($) |
| `ADMTG_DGNS_CD` | Admitting Diagnosis ICD-10 Code |
| `PRNCPAL_DGNS_CD` / `ICD_DGNS_CD1` | Principal Diagnosis ICD-10 Code |
| `REV_CNTR` | Revenue Center Code (`0450` = Emergency Room Services) |

### Step 4: Domain Field Categorization

- **Member Identifier Columns**: `BENE_ID`
- **Utilization-Related Fields**: `CLM_ID`, `CLM_FROM_DT`, `CLM_THRU_DT`, `CLM_ADMSN_DT`, `REV_CNTR` (`0450` for ED visits), `CLM_UTLZTN_DAY_CNT`
- **Diagnosis Fields**: `ADMTG_DGNS_CD`, `PRNCPAL_DGNS_CD`, `ICD_DGNS_CD1` through `ICD_DGNS_CD25`
- **Cost / Payment Fields**: `CLM_PMT_AMT`, `CLM_TOT_CHRG_AMT`, `NCH_BENE_IP_DDCTBL_AMT`, `NCH_BENE_PTB_DDCTBL_AMT`, `NCH_BENE_PTA_COINSRNC_LBLTY_AM`
- **Provider-Related Fields**: `PRVDR_NUM`, `ORG_NPI_NUM`, `AT_PHYSN_NPI`, `OP_PHYSN_NPI`, `OT_PHYSN_NPI`, `RNDRNG_PHYSN_NPI`

### Step 5: Analyze Inter-Dataset Relationships

In [ ]:
# Filter Emergency Room visits (REV_CNTR == '0450' or 450)
ed_inp = df_inp[df_inp['REV_CNTR'].astype(str).str.contains('0450|450', na=False)]
ed_outp = df_outp[df_outp['REV_CNTR'].astype(str).str.contains('0450|450', na=False)]

inp_ed_counts = ed_inp.groupby('BENE_ID').agg(ed_inpatient_visits=('CLM_ID', 'nunique'), total_inp_ed_spend=('CLM_PMT_AMT', 'sum')).reset_index()
outp_ed_counts = ed_outp.groupby('BENE_ID').agg(ed_outpatient_visits=('CLM_ID', 'nunique'), total_outp_ed_spend=('CLM_PMT_AMT', 'sum')).reset_index()

# Merge with Member Beneficiary File
bene_profile = df_bene[['BENE_ID', 'AGE_AT_END_REF_YR', 'SEX_IDENT_CD', 'STATE_CODE']].copy()
bene_profile = bene_profile.merge(inp_ed_counts, on='BENE_ID', how='left').merge(outp_ed_counts, on='BENE_ID', how='left')
bene_profile.fillna(0, inplace=True)
bene_profile['total_ed_visits'] = bene_profile['ed_inpatient_visits'] + bene_profile['ed_outpatient_visits']
bene_profile['total_ed_spend'] = bene_profile['total_inp_ed_spend'] + bene_profile['total_outp_ed_spend']

print("=== Beneficiary ED Utilization Profile ===")
print(f"Total Beneficiaries: {len(bene_profile):,}")
print(f"Beneficiaries with >= 1 ED Visit: {(bene_profile['total_ed_visits'] > 0).sum():,}")
print(f"Beneficiaries with >= 3 High-Frequent ED Visits: {(bene_profile['total_ed_visits'] >= 3).sum():,}")
bene_profile.head(10)

### Step 6: Downstream Model & View Feature Mapping

1. **XGBoost Risk Model**:
   - Input Features: `AGE_AT_END_REF_YR`, `SEX_IDENT_CD`, `STATE_CODE`, `total_ed_visits`, `ed_inpatient_visits`, `ed_outpatient_visits`, `total_ed_spend`.
   - Target Label: `high_risk_flag` (`total_ed_visits >= 3`).

2. **Isolation Forest Anomaly Detection**:
   - Features: Monthly visit acceleration rate (`CLM_FROM_DT` spikes), ratio of ED outpatient to inpatient claims, extreme single-encounter cost variance (`CLM_PMT_AMT`).
   - Output: Anomaly score, detected anomaly periods, and reason.

3. **Payer Analytics Dashboard**:
   - Metrics: Population counts, High-risk ratio, ED utilization trends over time, historical claim spend distribution (Potential Utilization Opportunity).

4. **Member History Timeline**:
   - Fields: `CLM_FROM_DT`, `CLM_ID`, `PRVDR_NUM`, `PRNCPAL_DGNS_CD`, `CLM_PMT_AMT`, facility type.